# Diffusion 学习
## 本文档是我的 Diffusion 学习笔记

我愿意从比 Diffusion 更远古的图像生成方案说起，也就是 Score Matching。这句话也许不严谨，因为 VAE 的历史似乎比 SM 更早。总之假设我仅仅了解机器学习的基础内容，从这里开始了解图像生成的庞大世界。

# Score Matching

### 目标函数与训练

生成图式模型本质在试图学习一个内容：在多维空间中，哪个点对应的真实图片概率最大。换言之，我们试图使用机器学习拟合真实图片在多维空间中的分布 $p_\text{data}$。如果我们得知了空间中哪个点对应猫，生成猫图片直接取出该点即可。这是美丽的想法。

为了学习图片分布 $p_\text{data}$，我们提出了 Score Matching。简而言之，我们不直接学习 $p_\text{data}$ 本身，而是学习空间中每个点处 $p_\text{data}$ 的梯度。我们不直接尝试拟合一座山，而是学习这座山每一处陡峭程度如何。为什么这样？请看

$$p(\mathbf{x}) = \frac{f(\mathbf{x})}{Z_\theta}, \quad Z_\theta = \int f(\mathbf{x}) d\mathbf{x}$$

上式中，$f(\mathbf{x})$ 实际上是能量函数。比如对于每个点，模型吐出一个分数，表示这个点是真实图片概率，这就是能量函数基本思想。但是 $p(\mathbf{x})$ 实际上需要归一化处理，因为全空间每点是真实图片概率的积分应该严格为 `1`。这里的问题就是这个归一化分母是积分 $\int f(\mathbf{x}) d\mathbf{x}$ ，而其无比棘手。但是转化成梯度就可以避免这个问题。请看

定义 $$\psi(\mathbf{x}) \equiv \nabla_{\mathbf{x}} \ln p(\mathbf{x})$$
那么 $$\nabla_{\mathbf{x}} \ln p(\mathbf{x}) = \nabla_{\mathbf{x}} \ln \left( \frac{f(\mathbf{x})}{Z_\theta} \right)$$
$$\nabla_{\mathbf{x}} \ln p(\mathbf{x}) = \nabla_{\mathbf{x}} \left( \ln f(\mathbf{x}) - \ln Z_\theta \right)$$
其中 $\nabla_{\mathbf{x}} \ln Z_\theta = 0$
得到 $$\psi(\mathbf{x}) = \nabla_{\mathbf{x}} \ln f(\mathbf{x})$$

这意味着，如果我们仅仅关注形状，那个归一化分母积分是无必要的。
如何训练？现在问题变成了：我们想要模型的坡度 $\mathbf{s}_\theta(\mathbf{x})$ 接近真实数据的坡度 $\nabla_{\mathbf{x}} \ln p_{\text{data}}(\mathbf{x})$。
我们定义目标函数 $$J(\theta) = \mathbb{E}_{p_{\text{data}}} \left[ \frac{1}{2} \| \mathbf{s}_\theta(\mathbf{x}) - \nabla_{\mathbf{x}} \ln p_{\text{data}}(\mathbf{x}) \|^2 \right]$$

问题在这里。我们不知道真实数据的坡度 $\nabla_{\mathbf{x}} \ln p_{\text{data}}(\mathbf{x})$，因为我们手里只有样本，没有数据分布的解析式。我们要绞尽脑汁采样，努力逼近这个分布，这很困难。但是有一个变换可以解决这个问题。请看

展开 $$J(\theta) = \mathbb{E}_{p_{\text{data}}} \left[ \frac{1}{2} \|\mathbf{s}_\theta(\mathbf{x})\|^2 - \mathbf{s}_\theta(\mathbf{x}) \cdot \nabla_{\mathbf{x}} \ln p_{\text{data}}(\mathbf{x}) + \frac{1}{2} \|\nabla_{\mathbf{x}} \ln p_{\text{data}}(\mathbf{x})\|^2 \right]$$

第一项$\frac{1}{2} \|\mathbf{s}_\theta(\mathbf{x})\|^2$，只包含模型，可直接计算。第三项$\frac{1}{2} \|\nabla_{\mathbf{x}} \ln p_{\text{data}}(\mathbf{x})\|^2$是个常数，在优化时（我们计算梯度）会被忽略。请关注交叉项，它包含了未知的 $\nabla_{\mathbf{x}} \ln p_{\text{data}}(\mathbf{x})$。

我们要处理的是交叉项的期望。为了清晰起见，我们对分量进行讨论。假设 $\mathbf{x}$ 是 $d$ 维的。根据向量乘法，$$\mathbf{a} \cdot \mathbf{b} = \sum_{i=1}^d a_i b_i$$ 我们有 $$\mathbb{E}_{p_{\text{data}}} [ \mathbf{s}_\theta(\mathbf{x}) \cdot \nabla_{\mathbf{x}} \ln p_{\text{data}}(\mathbf{x}) ] = \sum_{i=1}^d \int p_{\text{data}}(\mathbf{x}) s_{\theta, i}(\mathbf{x}) \frac{\partial \ln p_{\text{data}}(\mathbf{x})}{\partial x_i} d\mathbf{x}$$
你没懂的话，我再解释一下。这里实际上代换了 $$\mathbf{s}_\theta(\mathbf{x}) \cdot \nabla_{\mathbf{x}} \ln p_{\text{data}}(\mathbf{x}) = \sum_{i=1}^d s_{\theta, i}(\mathbf{x}) \left( \frac{\partial \ln p_{\text{data}}(\mathbf{x})}{\partial x_i} \right)$$
我们继续吧。根据$\frac{\partial \ln p}{\partial x} = \frac{1}{p} \frac{\partial p}{\partial x}$，我们有 $$\int p_{\text{data}}(\mathbf{x}) s_{\theta, i}(\mathbf{x}) \left( \frac{1}{p_{\text{data}}(\mathbf{x})} \frac{\partial p_{\text{data}}(\mathbf{x})}{\partial x_i} \right) d\mathbf{x} = \int s_{\theta, i}(\mathbf{x}) \frac{\partial p_{\text{data}}(\mathbf{x})}{\partial x_i} d\mathbf{x}$$
使用分部积分。 $$\int s_{\theta, i}(\mathbf{x}) \frac{\partial p_{\text{data}}(\mathbf{x})}{\partial x_i} d\mathbf{x} = \underbrace{\left[ s_{\theta, i}(\mathbf{x}) p_{\text{data}}(\mathbf{x}) \right]_{-\infty}^{+\infty}}_{= 0} - \int p_{\text{data}}(\mathbf{x}) \frac{\partial s_{\theta, i}(\mathbf{x})}{\partial x_i} d\mathbf{x}$$
注意，这里我们假设了$\|x\| \to \infty$ 时，$p_{\text{data}}(x) \to 0$。这个假设是合理的，因为真实语义空间或者像素空间不是无限扩张的。
将所有的分量 $i$ 重新加和，我们得到 $$-\sum_{i=1}^d \mathbb{E}_{p_{\text{data}}} \left[ \frac{\partial s_{\theta, i}(\mathbf{x})}{\partial x_i} \right] = -\mathbb{E}_{p_{\text{data}}} [ \text{div}(\mathbf{s}_\theta(\mathbf{x})) ] = -\mathbb{E}_{p_{\text{data}}} [ \text{tr}(\nabla_{\mathbf{x}} \mathbf{s}_\theta(\mathbf{x})) ]$$
这里的 $\nabla_{\mathbf{x}} \mathbf{s}_\theta(\mathbf{x})$ 是分数的雅可比矩阵（雅可比的迹就是散度 Divergence），如果 $\mathbf{s}_\theta$ 是某个能量函数的梯度，这一项就是 Hessian 矩阵的trace。

代回原始目标函数，得到 $$J(\theta) = \mathbb{E}_{p_{\text{data}}} \left[ \frac{1}{2} \|\mathbf{s}_\theta(\mathbf{x})\|^2 + \text{tr}(\nabla_{\mathbf{x}} \mathbf{s}_\theta(\mathbf{x})) \right] + \text{const}$$
真实环境中，数据分布只能使用蒙特卡洛估计。换言之，对于$$\mathbb{E}_{p_{\text{data}}} [g(\mathbf{x})] = \int p_{\text{data}}(\mathbf{x}) g(\mathbf{x}) d\mathbf{x}$$ 一个估计方法是采样 $N$ 个样本$\{x_1, x_2, \dots, x_N\}$，估计 $$\mathbb{E}_{p_{\text{data}}} [g(\mathbf{x})] \approx \frac{1}{N} \sum_{i=1}^N g(\mathbf{x}_i)$$
所以最后的公式是 $$J(\theta) \approx \frac{1}{N} \sum_{i=1}^N \left( \text{tr}(\nabla_{\mathbf{x}} \mathbf{s}_\theta(\mathbf{x}_i)) + \frac{1}{2} \| \mathbf{s}_\theta(\mathbf{x}_i) \|^2 \right)  + \text{const}$$

这很漂亮。所有的项你都可以通过训练数据本身或者模型预测结果得知。

### 推理

如何推理？我们假设我们已经训练得到了一个模型，实际上我们已经得到了空间中每个地方梯度。因此推理就像机器学习优化器，慢慢移动。
我们从一个随机噪声 $\mathbf{x}_0$ 开始，按下式进行迭代 $$\mathbf{x}_{t+1} = \mathbf{x}_t + \frac{\eta}{2} \mathbf{s}_\theta(\mathbf{x}_t) + \sqrt{\eta} \mathbf{z}_t, \quad \mathbf{z}_t \sim \mathcal{N}(0, I)$$
注意，这里初始的随机噪声一般就是高斯噪声。如果你还不知道高斯噪声在高维空间上的分布，你可以先去了解。这是比较关键的内容。

所以这里每个项含义是：$\mathbf{x}_t$ 是当前的图片状态（初始时是纯噪声）。$\frac{\eta}{2} \mathbf{s}_\theta(\mathbf{x}_t)$是确定性漂移。$\mathbf{s}_\theta(\mathbf{x}_t)$实际上就是预测的当前点梯度方向（$\eta$ 是步长）。$\sqrt{\eta} \mathbf{z}_t$是随机扰动。为了防止粒子直接掉进最近的一个死胡同（局部最大值），我们给它加一点抖动。

这个抖动实际上很重要，是 Langevin Dynamics 的核心部分。如果你还不知道 Langevin Dynamics 是什么，简单来说就是加点扰动可以收敛更快。

当步长 $\eta \to 0$ 且迭代次数 $T \to \infty$ 时，这种方法保证了最终得到的 $\mathbf{x}_T$ 是严格服从 $p_{\text{data}}(\mathbf{x})$ 分布的独立样本。

### 有条件训练与无条件训练

以上是 Score Matching 的原始训练方式和推理。理论上已经可以进行无条件训练和推理了。如果你还不知道什么是无条件，简单解释就是模型实际上不能既生成猫又生成狗，而是你训练时给他喂很多猫，他面对纯噪声就会逐渐推理出来猫的图片。这是因为我们推理时根本没告诉模型我们要猫还是狗（我们没做这个接口）。对于更先进的生成式模型，实际上推理还会接收一个条件参数 $c$，这个参数可以是自然语言或者就是一个数字，通过各种 Embedding 或者变换接入网络。我们简短说说有条件生成部分。

当我们要结合条件 $y$（比如文本“猫”）时，我们的目标分布从 $p(\mathbf{x})$ 变成了条件概率 $p(\mathbf{x}|y)$。
根据贝叶斯定理，$p(\mathbf{x}|y) = \frac{p(y|\mathbf{x})p(\mathbf{x})}{p(y)}$。
得到 $$\underbrace{\nabla_{\mathbf{x}} \ln p(\mathbf{x}|y)}_{\text{条件分数}} = \underbrace{\nabla_{\mathbf{x}} \ln p(\mathbf{x})}_{\text{无条件分数}} + \underbrace{\nabla_{\mathbf{x}} \ln p(y|\mathbf{x})}_{\text{分类器引导项}}$$

这里有两种方法来处理 $\underbrace{\nabla_{\mathbf{x}} \ln p(y|\mathbf{x})}_{\text{分类器引导项}}$ 这一项。

第一种是直接训练条件分数模型 (Conditional Score Network)。你不再训练 $\mathbf{s}_\theta(\mathbf{x})$，而是训练 $\mathbf{s}_\theta(\mathbf{x}, y)$。在输入神经网络时，除了图片 $\mathbf{x}$，还把条件 $y$（通常经过 Embedding，比如 CLIP 的 text encoder）作为一个额外的特征输入。

第二种方法会简单点，叫做分类器引导 (Classifier Guidance)。你不需要重新训练巨大的生成模型，只需要额外训练一个轻量级的分类器 $p(y|\mathbf{x})$（判断一张图有多像猫）。训练时生成模型只管画真实的图，分类器打分。推理时分类器接入模型前向传播，所以整个前向传播的权重会变化。分类器是可切换的，理论上切换到分类狗的分类器就可以生成狗。

不过以上两种方案都比较早期。实际上现在主流是做一个自由分类器 Classifier-Free Guidance (CFG)，原因是传统分类器处理效果不佳且麻烦。值得一提的是，自由分类器不是一个分类器，而是一种直接训练原始模型的做法，换言之更接近第一种。CFG 的做法是，我们在训练同一个模型时，有时给它条件 $y$，让它预测带条件的情况；有时不给条件（输入一个空的占位符 $\emptyset$），此时，模型实际上是在进行无条件训练，也就是在学 $p(x)$ 的分数。在推理时，我们用这两者的差值来强化语义：$$\mathbf{\hat{s}} = \mathbf{s}_\theta(\mathbf{x}, \emptyset) + \omega \left( \mathbf{s}_\theta(\mathbf{x}, y) - \mathbf{s}_\theta(\mathbf{x}, \emptyset) \right)$$
$\mathbf{s}_\theta(\mathbf{x}, y) - \mathbf{s}_\theta(\mathbf{x}, \emptyset)$ 这一项被认为是纯粹的语义向量，它剔除了背景噪声，只保留了从“无”到“猫”的演变方向。

实际上 CFG 绝对算很大的进步，证据是目前的 State-of-the-Art 都是采用的 CFG。大家发现模型原生就懂概率分布比训练两个模型稳定且简单得多。

你应该发现了，实际上训练有条件情况本质还是训练一个特定的无条件情况。因此我们回到原始的无条件训练。

### Score Matching 的问题与解决

原始 Score Matching 最大的问题就是$\text{tr}(\nabla_{\mathbf{x}} \mathbf{s}_\theta(\mathbf{x}))$计算开销不小，原因是计算 Hessian 矩阵的迹非常昂贵。如果 $\mathbf{x}$ 有 $10^6$ 个维度，计算迹需要对每个维度求偏导，这相当于要跑 $10^6$ 次反向传播。

推荐你去读 https://arxiv.org/abs/1907.05600  Generative Modeling by Estimating Gradients of the Data Distribution 原文。作者给出了两种解决问题的方法。首先是 Sliced Score Matching (切片分数匹配)，核心逻辑是利用随机投影（Hutchinson's Trick）来近似雅可比矩阵的迹。公式是 $$\mathbb{E}_{p_{\mathbf{v}}} \mathbb{E}_{p_{\text{data}}} [\mathbf{v}^T \nabla_{\mathbf{x}} \mathbf{s}_\theta(\mathbf{x}) \mathbf{v} + \frac{1}{2} \|\mathbf{s}_\theta(\mathbf{x})\|_2^2]$$

它不是去算整个雅可比矩阵，而是投射到一个随机方向 $\mathbf{v}$ 上。这样避开了 $D$ 次反向传播，但计算量依然不小。尽管解决了部分问题，这种方法依然没有得到很广泛的认可。另一种方法是则颠覆性的，名为 Denoising Score Matching (去噪分数匹配)。我们单开一节讲述这个内容。

# Denoising Score Matching

### 目标函数与训练

DSM 的核心思想是，与其直接去拟合真实的概率分布（$p_{data}$），不如先给真实分布加噪声$\sigma$，然后研究噪声长什么样。这已经有了去噪模型的雏形，或者换句话说就是初代去噪模型。

我们不再研究原始分布 $p_{data}(\mathbf{x})$，而是研究一个加了噪声的分布 $q_\sigma(\tilde{\mathbf{x}})$：$$q_\sigma(\tilde{\mathbf{x}}) = \int p_{data}(\mathbf{x}) \underbrace{q_\sigma(\tilde{\mathbf{x}}|\mathbf{x})}_{\text{加噪过程}} d\mathbf{x}$$
其中 $q_\sigma(\tilde{\mathbf{x}}|\mathbf{x}) = \mathcal{N}(\tilde{\mathbf{x}} | \mathbf{x}, \sigma^2 \mathbf{I})$。这就是说，每一个真点 $\mathbf{x}$ 都被扩散成了一个高斯云。

如果站在加噪点 $\tilde{\mathbf{x}}$ 上，想找回原点 $\mathbf{x}$，因为 $q_\sigma(\tilde{\mathbf{x}}|\mathbf{x})$ 是我们已知的（高斯分布），它的分数（对数梯度）是已知的：$$\nabla_{\tilde{\mathbf{x}}} \ln q_\sigma(\tilde{\mathbf{x}}|\mathbf{x}) = \nabla_{\tilde{\mathbf{x}}} \left( -\frac{\| \tilde{\mathbf{x}} - \mathbf{x} \|^2}{2\sigma^2} + \text{const} \right) = \frac{\mathbf{x} - \tilde{\mathbf{x}}}{\sigma^2}$$

请注意，根据 Score Matching 的定义，我们优化的就是模型预测与分数的距离。这一点我们在最初已经解释过，我们关心每点梯度而不是原本的模样。所以目标函数就是$$\mathcal{L}_{DSM}(\theta) = \frac{1}{2} \mathbb{E}_{p_{data}(\mathbf{x})} \mathbb{E}_{q_\sigma(\tilde{\mathbf{x}}|\mathbf{x})} \left[ \| \mathbf{s}_\theta(\tilde{\mathbf{x}}) - \frac{\mathbf{x} - \tilde{\mathbf{x}}}{\sigma^2} \|^2 \right]$$

实际上更有力的证据是 2011 年 Pascal Vincent 证明了以下等价性：优化上式这个预测噪声的目标函数在数学上完全等价于让模型去拟合加噪后分布 $q_\sigma(\tilde{\mathbf{x}})$ 的真实分数 $\nabla_{\tilde{\mathbf{x}}} \ln q_\sigma(\tilde{\mathbf{x}})$。所以我们没有逃离 Score Matching 的讨论范围。

### 推理

DSM 的推理非常近似于原始的 Score Matching。Score Matching 推理是 $\mathbf{x}_{new} = \mathbf{x}_{old} + \eta \mathbf{s}_\theta(\mathbf{x}_{old})$，而 DSM 的学习结果是 $\mathbf{s}_\theta(\tilde{\mathbf{x}}) \approx \frac{\mathbf{x} - \tilde{\mathbf{x}}}{\sigma^2}$。因此推理是 $$\mathbf{x}_{new} = \tilde{\mathbf{x}} + \eta \left( \frac{\mathbf{x} - \tilde{\mathbf{x}}}{\sigma^2} \right)$$
如果取 $\eta = \sigma^2$，你会发现：$\mathbf{x}_{new} = \tilde{\mathbf{x}} + (\mathbf{x} - \tilde{\mathbf{x}}) = \mathbf{x}$。

不过我们一般是不会使用这种推理方式的，原因是固定的推理极易坍缩到单点。原始论文中，作者给出的推理公式是 $$\underbrace{\tilde{\mathbf{x}}_t}_{\text{下一步}} = \underbrace{\tilde{\mathbf{x}}_{t-1}}_{\text{当前步}} + \underbrace{\frac{\epsilon}{2} \nabla_{\mathbf{x}} \log p(\tilde{\mathbf{x}}_{t-1})}_{\text{确定性漂移 (Drift)}} + \underbrace{\sqrt{\epsilon} \mathbf{z}_t}_{\text{随机扩散 (Diffusion)}}$$
其中确定性漂移就是 $\eta \mathbf{s}_\theta$。它提供了一个引力，强制粒子向概率密度最高的地方移动。这里 $\eta = \epsilon / 2$。
扩散项就是 $\sqrt{\epsilon} \mathbf{z}_t$。它是一个标准的布朗运动项，给粒子注入随机的能量，换言之，Langevin Dynamics。

至于系数为什么是 $\sqrt{\epsilon}$ 和 $\epsilon$，说来话长，这是 Fokker-Planck 方程的结果。

非常值得一提的是，Score Matching 和 DSM 实际上共享一套推理公式。更本质的结论是，这两套模型最终学习到的向量场也是一样的。SM 学习的是原始分布 $p_{\text{data}}$ 的分数 $\mathbf{s}_{\theta} \approx \nabla \log p_{\text{data}}(\mathbf{x})$ 。而 DSM 学习的是加噪分布 $q_\sigma$ 的分数 $\mathbf{s}_{\theta} \approx \nabla \log q_\sigma(\mathbf{x})$

当噪声 $\sigma \to 0$ 时，加噪分布 $q_\sigma$ 会通过 Dirac 测度坍缩回原始分布 $p_{\text{data}}$。$$\lim_{\sigma \to 0} q_\sigma(\mathbf{x}) = p_{\text{data}}(\mathbf{x})$$因此，在噪声趋于零的极限下，两者学习到的向量场在数学上是完全恒等的。

但在现实中，$\sigma$ 不可能为 0。这导致了 SM 和 DSM 本质的不同。

SM 的向量场长相锐利但破碎，只在数据点附近有极强的引力，一旦粒子稍微飘远，进入高维空间的荒漠，向量场就会因为没有受过训练而变得随机且混乱。DSM 的向量场长相模糊但稳健。由于 $q_\sigma$ 是原始分布与高斯核的卷积 ($p_{\text{data}} * \mathcal{N}(0, \sigma^2)$)，DSM 实际上学习的是一个平滑后的地形图。即使粒子离数据流形很远，它也能感受到一股温和的引力指向数据区。

不过 DSM 的代价是，它学到的梯度不够精确。如果你用较大的 $\sigma$ 练出的模型去生成，图片会显得模糊，像被打了一层柔光滤镜

### DSM 的问题与解决

一个著名的**流形假设** (Manifold Hypothesis) 是，现实世界中的数据往往集中于嵌入高维空间（即环境空间）中的低维流形。该假设在许多数据集中经验成立，并成为流形学习的基础。如果你不知道什么是流形，那么你绝对应该去了解一下。简单来说，流行是对更抽象空间的描述。

在流形假设下，数据分布 $p_{data}$ 在高维空间中仅仅占据极小的空间，比如巨大三维空间中的一张纸。在 $p_{data}$ 以外的地方概率为 0，对数梯度 $\nabla \log 0$ 是数学上的灾难。不过这是传统 SM 的严重问题却不是 DSM 的问题，原因是 DSM 不学习 $p_{data}$，它学习的是 $q_\sigma = p_{data} * \mathcal{N}(0, \sigma^2)$。这就好像给原本没有内容的区域加上简单的雾。

但是这并不代表着 DSM 完全解决了流形假设中低维流形稀有的问题。实际上，在低密度区域的信号依旧微弱，如果只用一个微小的噪声 $\sigma$ 来训练 DSM，虽然数学上全空间都拥有了梯度，但离流形稍微远一点的地方，梯度模长会呈指数级衰减。Langevin Dynamics 通常从高斯噪声开始，粒子感受到的引力微乎其微，它可能需要迭代非常非常久才能摸到流形的边缘。

困难不止这些。在 Generative Modeling by Estimating Gradients of the Data Distribution 原论文中，作者指出了其他挑战。如采样低密度数据集部分的困难。在数据密度原本就低下的部分，模型难以学习到正确的梯度方向。其次是 Langevin Dynamics 本身的著名混合问题 (Mixing Problem)。简单来说就是粒子非常难以从一个高引力区域走向另一个高引力区域，即使后者正是我们所需的。我们来详细解释这个问题。

根据 Langevin Dynamics 的更新方式$$d\mathbf{x} = \frac{1}{2} \nabla \ln p(\mathbf{x}) dt + d\mathbf{w}$$ 粒子的更新实际上受确定性力 $\nabla \ln p$ 和 随机力 (布朗运动) $d\mathbf{w}$ 驱使。当粒子处于 Mode A 处，这里是真实图片，引力非常强。如果它想迭代到 Mode B，必须先经过中间的荒漠。由于荒漠里梯度混乱且稀疏，这里几乎无法穿越。

你可能会考虑前面提到的有条件训练下的分类器，实际上分类器自己在荒漠区域也很无能。荒漠区域的原罪是数据样本太少，导致模型在这个地方的梯度基本都是混乱的。更具体的，我们展开刚刚所说的确定性力部分$$\nabla_{\mathbf{x}} \ln p(\mathbf{x}|y) = \nabla_{\mathbf{x}} \ln p(\mathbf{x}) + \omega \nabla_{\mathbf{x}} \ln p(y|\mathbf{x})$$
请注意，现在确定性力部分变成有条件的，也就是左式。

无条件分数项 $\nabla_{\mathbf{x}} \ln p(\mathbf{x})$ 始终在把粒子拉向任何它认为真实的地带，因此 ModeA 引力很强。另外的，在 ModeA 与 ModeB之间的真空地带，$\nabla_{\mathbf{x}} \log p(y|\mathbf{x})$ 本身也不一定能够给出正确的梯度。由此看见，分类器不解决本质问题。

解决上述一切问题的核心思想如下：在荒漠区域，我们需要大噪声 $\sigma_{high}$ 来帮助粒子快速接近真正具有价值的低维流形；在低维流形中，我们需要小噪声 $\sigma_{low}$ 来提升推理质量。这里的思想很像机器学习优化器中的 Adam，在梯度平缓地区加大优化步长，在梯度陡峭地区减小步长。如果你不知道什么 Adam，这几乎不可能，我比较难以相信你居然在阅读关于 Diffusion 的笔记。

遵循上述的思想，我们介绍 Noise Conditional Score Networks (噪声条件分数网络)。我们再开一节。

# Noise Conditional Score Networks

### 目标函数与训练

我们预定义一个单调递减的噪声序列：$$\sigma_1 > \sigma_2 > \dots > \sigma_L$$
其中，$\sigma_1$（极大噪声）是大到足以填平所有的低密度真空地区，连通所有的孤立模态，从而解决混合问题的噪声。$\sigma_L$（极小噪声）则是小到几乎不影响原始数据的细节，保证生成的清晰度。

每一个噪声级别对应一个分布 $p_{\sigma_i}(\mathbf{x}) = \int p_{data}(\mathbf{y}) \mathcal{N}(\mathbf{x}; \mathbf{y}, \sigma_i^2 \mathbf{I}) d\mathbf{y}$。

我们需要修改模型结构。NCSN 并不是训练 $L$ 个模型，而是训练一个共享参数的神经网络 $\mathbf{s}_\theta(\mathbf{x}, \sigma)$。输入是当前的图片 $\mathbf{x}$ 和当前的噪声强度 $\sigma$（通常将 $\sigma$ 编码后喂给网络）。输出则对应当前噪声级别的分数估计。

对于每一个尺度 $\sigma_i$，NCSN 使用 Denoising Score Matching (DSM) 进行训练。最终的 Loss 是所有尺度损失的加权和：$$\mathcal{L}(\theta; \{\sigma_i\}_{i=1}^L) = \frac{1}{L} \sum_{i=1}^L \lambda(\sigma_i) \mathbb{E}_{p_{data}(\mathbf{x})} \mathbb{E}_{q_{\sigma_i}(\tilde{\mathbf{x}}|\mathbf{x})} \left[ \| \mathbf{s}_\theta(\tilde{\mathbf{x}}, \sigma_i) - \frac{\mathbf{x} - \tilde{\mathbf{x}}}{\sigma^2} \|^2 \right]$$一个关键的工程细节是$\lambda(\sigma_i) = \sigma_i^2$。原因是当模型训练得比较好时，$\| \mathbf{s}_\theta \|$ 的量级大约与 $1/\sigma$ 成正比。通过令权重 $\lambda(\sigma_i) = \sigma_i^2$，可以让所有噪声量级下的 Loss 处于同一个数值量级。更具体的，真实的 Score（对数梯度）量级大约是 $\frac{\mathbf{x} - \tilde{\mathbf{x}}}{\sigma^2}$。由于 $\mathbf{x} - \tilde{\mathbf{x}}$ 本质上是高斯噪声 $\sigma \epsilon$，所以：$$\| \mathbf{s}_\theta(\tilde{\mathbf{x}}, \sigma_i) - \frac{\mathbf{x} - \tilde{\mathbf{x}}}{\sigma^2} \|^2 \approx \| \frac{\sigma \epsilon}{\sigma^2} \|^2 = \frac{\| \epsilon \|^2}{\sigma^2} \propto \frac{1}{\sigma^2}$$因此乘上 $\sigma_i^2$ 之后，这一项会被归一化。

实际训练中，我们是依旧采样估计真实的概率分布。一个 Batch 的训练逻辑通常是这样的：从数据集里采样一个 Batch 的干净图片 $\{\mathbf{x}^{(1)}, \dots, \mathbf{x}^{(n)}\}$。为这个 Batch 里的每一张图片，随机从 $\{1, \dots, L\}$ 中抽一个索引 $i$。这意味着，同一个 Batch 里，图 A 可能被加了巨大的噪声 $\sigma_1$，而图 B 可能只加了微弱的噪声 $\sigma_L$。根据各自抽到的 $\sigma_i$，生成脏图 $\tilde{\mathbf{x}}$ $$\tilde{\mathbf{x}} = \mathbf{x} + \sigma_i \mathbf{\epsilon}$$再把 $\tilde{\mathbf{x}}$ 和对应的 $\sigma_i$ 给模型计算 $s_\theta(\tilde{\mathbf{x}}, \sigma_i)$，再计算当前这一对 $(\tilde{\mathbf{x}}, \sigma_i)$ 的 DSM Loss$$\mathcal{L} = \sigma_i^2 \cdot \| \mathbf{s}_\theta(\tilde{\mathbf{x}}, \sigma_i) + \frac{\mathbf{\epsilon}}{\sigma_i} \|^2$$

再计算完一整个 Batch 的 Loss 之后，相加取平均，反向传播并且执行梯度更新。所以实际上理想的 $\sum_{i=1}^L$（累加符号），在随机梯度下降（SGD）中被转化为在足够多次的迭代后，模型会均匀地遍历到所有的噪声级。

值得一提的是，NSCN 原论文中指出推理不直接使用最新的参数，而是使用被维护的 EMA 参数$$\theta_{ema} \leftarrow \mu \cdot \theta_{ema} + (1 - \mu) \cdot \theta_{current}$$这很容易理解，很多模型都会使用这个工程化技巧保证变化不至于过度剧烈。$\mu$ 一般取 $0.999$。

### 推理

正式采样生成图时，NCSN 模拟了一个“从混沌到秩序”的过程。其算法逻辑如下：

初始化：从纯噪声 $\mathbf{x}_0$（或者覆盖全空间的简单分布）开始。

迭代（外循环）：从 $i=1$ 到 $L$ 遍历每一个噪声级。

采样（内循环）：在当前的 $\sigma_i$ 下，运行 $T$ 步朗之万动力学迭代：$$\mathbf{x}_t = \mathbf{x}_{t-1} + \eta_i \mathbf{s}_\theta(\mathbf{x}_{t-1}, \sigma_i) + \sqrt{2\eta_i} \mathbf{z}_t$$这里的步长 $\eta_i$ 随 $\sigma_i$ 的减小而减小。通常取 $\eta_i \propto \sigma_i^2$，或者直接就是$\epsilon \cdot \sigma_i^2 / \sigma_L^2$。

传递：将 $\sigma_i$ 下得到的最终样本，作为 $\sigma_{i+1}$ 的初始位置。

以上的推理被称为 Annealed Langevin Dynamics。我们应该已经说过了，这样的训练和推理流程破解了流形假设中的稀疏问题，现在任何原本真空的区域理应被赋予一个合理的梯度。

### NSCN的问题与解决

NSCN 在各大数据集上取得了不错的成绩，但是其已经是非常早期的架构了。站在现在回望，我们发现 NSCN 最大的问题就是推理步骤太长。如果 $L=1000$，$T=5$，那么生成一张图需要运行 5000 次 神经网络前向计算，这实际上对于日益扩张的大模型参数是完全的灾难。现在的生成式模型一般会在 10-20 步之内得到最终结果。在下一个 Chapter，我们会介绍解决或者设法解决这些问题的架构。

# 总结

本章介绍了 Score Matching，Denosing Score Matching 与 Noise Conditional Score Networks 的主要内容。值得一提的是，本章实际上完全围绕 https://arxiv.org/abs/1907.05600  Generative Modeling by Estimating Gradients of the Data Distribution 原论文展开，因此推荐你去读。下一章或下下章我们会介绍 DDPM，DDIM 与 DPM Solver++ 等等更进一步的 Diffusion 架构，或者说正式引入 Diffusion 这个概念。一个预告是，Flow Matching 应该会在最后登场，这是目前的生成式模型的答案。